In [2]:
import cv2

def track_movements():
    """
    Detects and tracks any general movement using the webcam 
    via the Frame Differencing technique and records the video feed.
    """
    print("Starting webcam for motion tracking...")
    print("IMPORTANT: A new window will open. Press 'q' on your keyboard to quit.")
    
    # 1. Initialize the webcam (0 is the default camera)
    cap = cv2.VideoCapture(0)

    # --- NEW: Set up the VideoWriter to save the recording ---
    # Get the default frame width and height from the webcam
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Define the codec and create a VideoWriter object
    # 'XVID' is a very standard and reliable codec for .avi files
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    # Save as 'motion_recording.avi' at 20 Frames Per Second (FPS)
    out = cv2.VideoWriter('motion_recording.avi', fourcc, 20.0, (frame_width, frame_height))
    print("Recording started. Saving session to 'motion_recording.avi'...")
    # ---------------------------------------------------------

    # 2. Read the first two initial frames
    # We need two frames to compare them and find the differences
    ret, frame1 = cap.read()
    ret, frame2 = cap.read()

    if not ret:
        print("Error: Could not access the webcam.")
        return

    while cap.isOpened():
        # 3. Find the absolute difference between the two frames
        # Anything that hasn't moved will turn black (difference of 0)
        # Anything that moved will have a pixel difference > 0
        diff = cv2.absdiff(frame1, frame2)
        
        # 4. Convert the difference to grayscale (easier to process)
        gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        
        # 5. Apply a Gaussian Blur to smooth out the image and remove noise/static
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        
        # 6. Apply a Threshold
        # Any pixel difference greater than 20 becomes pure white (255)
        # Any difference less than 20 becomes pure black (0)
        _, thresh = cv2.threshold(blur, 20, 255, cv2.THRESH_BINARY)
        
        # 7. Dilate the image to make the white moving spots larger and fill in holes
        dilated = cv2.dilate(thresh, None, iterations=3)
        
        # 8. Find the contours (outlines) of the white shapes (the moving objects)
        contours, _ = cv2.findContours(dilated, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        
        movement_detected = False

        # 9. Draw rectangles around the moving objects
        for contour in contours:
            # Ignore tiny movements (like shadows or camera static) by checking the area size
            if cv2.contourArea(contour) < 1500:
                continue
            
            movement_detected = True
            
            # Get coordinates for the bounding box
            (x, y, w, h) = cv2.boundingRect(contour)
            
            # Draw the green rectangle on the original frame1
            cv2.rectangle(frame1, (x, y), (x+w, y+h), (0, 255, 0), 2)
            
        # Display an alert text if movement is happening
        if movement_detected:
            cv2.putText(frame1, "Status: MOVEMENT DETECTED!", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 3)
        else:
            cv2.putText(frame1, "Status: All Clear", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # --- NEW: Save the current frame to our video file ---
        out.write(frame1)
        # -----------------------------------------------------

        # 10. Show the final image to the user
        cv2.imshow('Security Feed - Motion Tracker', frame1)
        
        # 11. Shift the frames forward to continue the loop
        frame1 = frame2
        ret, frame2 = cap.read()
        
        # Wait 10ms, and break the loop if the 'q' key is pressed
        if cv2.waitKey(10) & 0xFF == ord('q'):
            print("Quitting motion tracking...")
            break

    # Clean up and release the camera AND the video writer
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("Webcam and recording released safely. Check your folder for 'motion_recording.avi'.")

# Run the function
if __name__ == "__main__":
    track_movements()

Starting webcam for motion tracking...
IMPORTANT: A new window will open. Press 'q' on your keyboard to quit.
Recording started. Saving session to 'motion_recording.avi'...
Quitting motion tracking...
Webcam and recording released safely. Check your folder for 'motion_recording.avi'.
